<a href="https://colab.research.google.com/github/Roshanraj2580/Rice_Leaf_Detection/blob/main/strict_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install ImageHash

import os
import shutil
import random
from pathlib import Path
from collections import defaultdict
from PIL import Image
import imagehash


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.1 MB/s eta 0:00:00


In [3]:
source_dir = Path("/content/drive/MyDrive/Minor_Project/Data/Processed")
clean_dir = Path("/content/drive/MyDrive/Minor_Project/Data/Processed_Cleaned")
split_base = Path("/content/drive/MyDrive/Minor_Project/Data/Splitted_Strict")

random.seed(42)


In [4]:
if clean_dir.exists():
    shutil.rmtree(clean_dir)

if split_base.exists():
    shutil.rmtree(split_base)

clean_dir.mkdir(parents=True, exist_ok=True)
split_base.mkdir(parents=True, exist_ok=True)

print("Fresh folders created.")


Fresh folders created.


In [5]:
removed_summary = {}

for class_dir in sorted(source_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name
    target_class_dir = clean_dir / class_name
    target_class_dir.mkdir(parents=True, exist_ok=True)

    kept_hashes = []
    kept_count = 0
    removed_count = 0

    for img_path in sorted(class_dir.glob("*")):
        if not img_path.is_file():
            continue

        try:
            img = Image.open(img_path).convert("RGB")
            ph = imagehash.phash(img)

            duplicate_found = False
            for old_hash in kept_hashes:
                if ph - old_hash <= 3:
                    duplicate_found = True
                    break

            if duplicate_found:
                removed_count += 1
            else:
                shutil.copy2(img_path, target_class_dir / img_path.name)
                kept_hashes.append(ph)
                kept_count += 1

        except Exception as e:
            print(f"Skipped {img_path.name}: {e}")

    removed_summary[class_name] = {
        "kept": kept_count,
        "removed": removed_count
    }

print("Duplicate cleaning completed.")


Duplicate cleaning completed.


In [6]:
for class_name, stats in removed_summary.items():
    print(f"{class_name}: kept={stats['kept']}, removed={stats['removed']}")


Bacterialblight: kept=515, removed=1069
Blast: kept=484, removed=956
Brownspot: kept=614, removed=986
Tungro: kept=833, removed=475


In [7]:
def get_group_id(filename):
    name = Path(filename).stem
    return name.split("_")[0]

grouped_data = {}

for class_dir in sorted(clean_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name
    groups = defaultdict(list)

    for img_path in sorted(class_dir.glob("*")):
        if img_path.is_file():
            group_id = get_group_id(img_path.name)
            groups[group_id].append(img_path)

    grouped_data[class_name] = groups

for class_name, groups in grouped_data.items():
    print(f"{class_name}: {len(groups)} groups")


Bacterialblight: 6 groups
Blast: 6 groups
Brownspot: 6 groups
Tungro: 5 groups


In [8]:
def get_group_id(filename):
    name = Path(filename).stem
    return name.split("_")[0]

grouped_data = {}

for class_dir in sorted(clean_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name
    groups = defaultdict(list)

    for img_path in sorted(class_dir.glob("*")):
        if img_path.is_file():
            group_id = get_group_id(img_path.name)
            groups[group_id].append(img_path)

    grouped_data[class_name] = groups


In [9]:
import shutil
import random
from pathlib import Path

train_dir = split_base / "train"
val_dir = split_base / "val"
test_dir = split_base / "test"

if split_base.exists():
    shutil.rmtree(split_base)

train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

split_summary = {}

for class_name, groups in grouped_data.items():
    group_ids = list(groups.keys())
    random.shuffle(group_ids)

    total_groups = len(group_ids)

    if total_groups < 3:
        print(f"Skipping {class_name}: not enough groups")
        continue

    train_count = max(1, int(round(total_groups * 0.70)))
    val_count = max(1, int(round(total_groups * 0.15)))
    test_count = total_groups - train_count - val_count

    if test_count < 1:
        test_count = 1
        if train_count > 1:
            train_count -= 1
        elif val_count > 1:
            val_count -= 1

    while train_count + val_count + test_count > total_groups:
        if train_count > val_count and train_count > 1:
            train_count -= 1
        elif val_count > 1:
            val_count -= 1
        else:
            break

    train_groups = group_ids[:train_count]
    val_groups = group_ids[train_count:train_count + val_count]
    test_groups = group_ids[train_count + val_count:train_count + val_count + test_count]

    split_summary[class_name] = {
        "train_groups": len(train_groups),
        "val_groups": len(val_groups),
        "test_groups": len(test_groups),
        "train_images": sum(len(groups[g]) for g in train_groups),
        "val_images": sum(len(groups[g]) for g in val_groups),
        "test_images": sum(len(groups[g]) for g in test_groups),
    }

    for split_name, selected_groups in [
        ("train", train_groups),
        ("val", val_groups),
        ("test", test_groups)
    ]:
        split_class_dir = split_base / split_name / class_name
        split_class_dir.mkdir(parents=True, exist_ok=True)

        for gid in selected_groups:
            for img_path in groups[gid]:
                shutil.copy2(img_path, split_class_dir / img_path.name)

print("Strict grouped split completed.")


Strict grouped split completed.


In [10]:
for class_name, stats in split_summary.items():
    print(f"\nClass: {class_name}")
    print(f"Train Groups : {stats['train_groups']}")
    print(f"Val Groups   : {stats['val_groups']}")
    print(f"Test Groups  : {stats['test_groups']}")
    print(f"Train Images : {stats['train_images']}")
    print(f"Val Images   : {stats['val_images']}")
    print(f"Test Images  : {stats['test_images']}")



Class: Bacterialblight
Train Groups : 4
Val Groups   : 1
Test Groups  : 1
Train Images : 305
Val Images   : 206
Test Images  : 4

Class: Blast
Train Groups : 4
Val Groups   : 1
Test Groups  : 1
Train Images : 323
Val Images   : 1
Test Images  : 160

Class: Brownspot
Train Groups : 4
Val Groups   : 1
Test Groups  : 1
Train Images : 606
Val Images   : 4
Test Images  : 4

Class: Tungro
Train Groups : 3
Val Groups   : 1
Test Groups  : 1
Train Images : 505
Val Images   : 156
Test Images  : 172


In [11]:
import os

base = "/content/drive/MyDrive/Minor_Project/Data/Splitted_Strict"

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    split_path = os.path.join(base, split)
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            print(f"{cls}: {count}")



TRAIN
Bacterialblight: 305
Blast: 323
Brownspot: 606
Tungro: 505

VAL
Bacterialblight: 206
Blast: 1
Brownspot: 4
Tungro: 156

TEST
Bacterialblight: 4
Blast: 160
Brownspot: 4
Tungro: 172


In [12]:
for class_name, stats in split_summary.items():
    print(f"\nClass: {class_name}")
    print(f"Train Groups : {stats['train_groups']}")
    print(f"Val Groups   : {stats['val_groups']}")
    print(f"Test Groups  : {stats['test_groups']}")
    print(f"Train Images : {stats['train_images']}")
    print(f"Val Images   : {stats['val_images']}")
    print(f"Test Images  : {stats['test_images']}")



Class: Bacterialblight
Train Groups : 4
Val Groups   : 1
Test Groups  : 1
Train Images : 305
Val Images   : 206
Test Images  : 4

Class: Blast
Train Groups : 4
Val Groups   : 1
Test Groups  : 1
Train Images : 323
Val Images   : 1
Test Images  : 160

Class: Brownspot
Train Groups : 4
Val Groups   : 1
Test Groups  : 1
Train Images : 606
Val Images   : 4
Test Images  : 4

Class: Tungro
Train Groups : 3
Val Groups   : 1
Test Groups  : 1
Train Images : 505
Val Images   : 156
Test Images  : 172


In [13]:
import os

print("Train exists:", os.path.exists("/content/drive/MyDrive/Minor_Project/Data/Splitted_Strict/train"))
print("Val exists  :", os.path.exists("/content/drive/MyDrive/Minor_Project/Data/Splitted_Strict/val"))
print("Test exists :", os.path.exists("/content/drive/MyDrive/Minor_Project/Data/Splitted_Strict/test"))


Train exists: True
Val exists  : True
Test exists : True


In [14]:
import os

base_path = "/content/drive/MyDrive/Minor_Project/Data"
print(os.listdir(base_path))


['Raw', 'Processed ', 'Processed', 'Splitted', 'Splitted_Grouped', 'Processed_Cleaned', 'Splitted_Strict']
